# Python Strings — From Basics to CPython Internals (+ Coding Questions)

> **Who this is for:** learners who know variables, loops and basic types and want to *really* understand strings — how to use them, how they live in memory, why they behave the way they do, and how to solve interview-style string problems.
>
> **Python version used for all outputs:** CPython 3.13. Exact byte sizes can differ slightly between versions; the *ideas* do not.


> **How to use this notebook:** run cells top to bottom (Kernel → Restart & Run All). The first code cell makes every expression in a cell display its result, so you can see each line's output.

---

## Table of Contents

1. [What a string is](#1-what-a-string-is)
2. [Creating strings](#2-creating-strings)
3. [Indexing and slicing](#3-indexing-and-slicing)
4. [Immutability](#4-immutability)
5. [Operators on strings](#5-operators-on-strings)
6. [String methods — the toolbox](#6-string-methods--the-toolbox)
7. [Formatting strings](#7-formatting-strings)
8. [Escape sequences and raw strings](#8-escape-sequences-and-raw-strings)
9. [Unicode, code points and encodings](#9-unicode-code-points-and-encodings)
10. [Internals: how CPython stores a string](#10-internals-how-cpython-stores-a-string)
11. [Internals: interning and identity](#11-internals-interning-and-identity)
12. [Internals: hashing](#12-internals-hashing)
13. [Internals: compile-time constant folding](#13-internals-compile-time-constant-folding)
14. [Performance: building strings efficiently](#14-performance-building-strings-efficiently)
15. [Time complexity cheat sheet](#15-time-complexity-cheat-sheet)
16. [Common pitfalls](#16-common-pitfalls)
17. [Coding questions — Easy](#17-coding-questions--easy)
18. [Coding questions — Medium](#18-coding-questions--medium)
19. [Coding questions — Hard](#19-coding-questions--hard)
20. [Concept / interview Q&A](#20-concept--interview-qa)
21. [Summary](#21-summary)

---

## 1. What a string is

A Python string (`str`) is an **immutable, ordered sequence of Unicode code points**.

Break that sentence down:

| Word | Meaning | Consequence |
|---|---|---|
| **immutable** | Once created, it can never change | Every "modification" creates a *new* string |
| **ordered** | Characters have positions | You can index (`s[0]`) and slice (`s[1:4]`) |
| **sequence** | Behaves like other sequences | `len()`, `in`, `for` loops, unpacking all work |
| **Unicode code points** | Each "character" is a number from the Unicode table | `"A"` is 65, `"€"` is 8364, `"😀"` is 128512 |

In [2]:
s = "Python"
print(type(s),len(s))

<class 'str'> 6


> **Key idea:** Python has **no separate `char` type**. A single character is just a string of length 1.

---

## 2. Creating strings

In [3]:
a = 'single quotes'
b = "double quotes"            # identical to single quotes
c = """triple quotes
can span
multiple lines"""
d = '''also triple'''
e = str(42)                    # convert another object -> '42'
f = ""                         # empty string (falsy)

**Why both `'` and `"`?** So you can put one kind inside the other without escaping:

In [4]:
msg = "It's easy"
html = '<div class="box">'

**Implicit concatenation** — adjacent literals are joined at *compile time*:

In [5]:
long_text = ("This is part one, "
             "and this is part two.")

In [6]:
print(long_text)

This is part one, and this is part two.


**Truthiness:**

In [8]:
bool("")      # False
bool(" ")     # True  (a space is a character!)

True

---

## 3. Indexing and slicing

### Visual: positive and negative indexes

```
  index:    0    1    2    3    4    5
          +----+----+----+----+----+----+
  s =     | P  | y  | t  | h  | o  | n  |
          +----+----+----+----+----+----+
  neg:     -6   -5   -4   -3   -2   -1
```

In [9]:
s = 'python'
print(s[0])

p


In [10]:
print(s[-1])

n


In [12]:
print(s[len(s)//2])

h


In [13]:
print(s[(len(s)//2) - 1])

t


In [14]:
s[10]

IndexError: string index out of range

### Slicing: `s[start:stop:step]`

- `start` is **included**, `stop` is **excluded**.
- Missing values default to "the whole way".
- Slicing **never raises IndexError** — it clamps to the valid range.

```
s[1:4]  ->  positions 1,2,3

          +----+====+====+====+----+----+
          | P  | y  | t  | h  | o  | n  |
          +----+====+====+====+----+----+
               ^start          ^stop (not included)
```

In [26]:
s = "Python"
s[1:4]     # 'yth'
s[:3]      # 'Pyt'
s[3:]      # 'hon'
s[:]       # 'Python'  (full copy — for str it actually returns the SAME object, see §11)
s[::2]     # 'Pto'     (every 2nd char)
s[::-1]    # 'nohtyP'  (reverse)
s[-3:]     # 'hon'     (last 3)
s[2:100]   # 'thon'    (no error)

'thon'

## 4. Immutability

In [27]:
s = "Venkata"
s[0] = "Babburi "

TypeError: 'str' object does not support item assignment

To "change" a string, build a new one:

In [28]:
s = "Babburi " + s

In [29]:
print(s)

Babburi Venkata


### Visual: what really happens

```
Before:   s ──► [ "cat" ]

After:    s ──► [ "bat" ]        [ "cat" ]  ◄── no names left → freed
```

### Why are strings immutable?

1. **Hashable** → can be dict keys and set members (a key's hash must never change).
2. **Safe to share** → many names can point to the same string without risk.
3. **Interning / caching** is possible (see §11).
4. **Thread-safe** reads without locks.

---

## 5. Operators on strings

| Operator | Example | Result |
|---|---|---|
| `+` concatenation | `"ab" + "cd"` | `'abcd'` |
| `*` repetition | `"py" * 3` | `'pypypy'` |
| `in` membership (substring) | `"a" in "cat"` | `True` |
| `not in` | `"z" not in "cat"` | `True` |
| `==`, `!=` | `"a" == "a"` | `True` |
| `<`, `>` (lexicographic by code point) | `"apple" < "banana"` | `True` |

In [30]:
"abc" + "def" 

'abcdef'

In [31]:
"ab" * 3

'ababab'

In [32]:
"s" in "sai vardhana"

True

In [33]:
"n" not in "Venkata"

False

In [36]:
s = "Python"
a = "Python"
print(id(s), id(a)) #--- shared object refernce
print(s == a)
print(s != a)

1531251245152 1531251245152
True
False


In [37]:
"Z" < "a"      # True — 'Z' is 90, 'a' is 97 (uppercase sorts before lowercase)
"10" < "9"     # True — compares '1' vs '9' character by character

True

In [39]:
name = "Sai Vardhan"
for ch in name:
    print(ch, ord(ch))

S 83
a 97
i 105
  32
V 86
a 97
r 114
d 100
h 104
a 97
n 110


---

## 6. String methods — the toolbox

All methods **return new strings** (or other values). None modify the original.

### 6.1 Case

In [49]:
"hello world".upper()       # 'HELLO WORLD'
"HELLO".lower()             # 'hello'
"hello world".title()       # 'Hello World'
"hello".capitalize()        # 'Hello'
"Hello".swapcase()          # 'hELLO'
"Straße".casefold()         # 'strasse'  ← best for case-insensitive comparison

'strasse'

### 6.2 Searching

In [54]:
s = "banana"
print(s.find("ana"))     #first index
print(s.find("anz"))    # -1 if missing

1
-1


In [55]:
print(s.rfind("an"))

3


In [57]:
print(s.index("an"))
print(s.index("anz"))

1


ValueError: substring not found

In [59]:
print(s.count("an"))
print(s.count("ana"))

2
1


In [61]:
print(s.startswith("ba"))
print(s.startswith("an"))

True
False


In [64]:
print(s.endswith("ba"))
print(s.endswith("na"))

False
True


### 6.3 Cleaning

In [80]:
"  hi  ".strip()          # 'hi'
"  hi  ".lstrip()         # 'hi  '
"xxhixx".strip("x")       # 'hi'   (strips a SET of chars, not a substring!)
"data.tar.gz".removesuffix(".gz")   # 'data.tar'  (3.9+)
"v1.2".removeprefix("v")            # '1.2'

'1.2'

### 6.4 Splitting and joining

In [82]:
"a,b,,c".split(",")

['a', 'b', '', 'c']

In [83]:
my_name = "Venkata Siva Sai Vardhan Babburi"
my_name.split()

['Venkata', 'Siva', 'Sai', 'Vardhan', 'Babburi']

In [89]:
my_name = "Venkata Siva Sai Vardhan Babburi"
name = my_name.split()
print(name)
new_name = ' '.join(name)
print(new_name)

['Venkata', 'Siva', 'Sai', 'Vardhan', 'Babburi']
Venkata Siva Sai Vardhan Babburi


> `join` is called on the **separator**, and it only accepts strings: `",".join([1, 2])` → TypeError. Use `",".join(map(str, [1, 2]))`.

### 6.5 Replacing and translating

In [90]:
"banana".replace("a",'o')

'bonono'

In [91]:
"banana".replace("a",'o',2)

'bonona'

In [92]:
table = str.maketrans("abc", "xyz", "d")
"abcd".translate(table)

'xyz'

### 6.6 Testing (return bool)

| Method | True when… |
|---|---|
| `isalpha()` | all letters |
| `isdigit()` | all digits (incl. some Unicode digits like `'١'`) |
| `isdecimal()` | strict decimal digits (safest for `int()` input) |
| `isnumeric()` | digits + things like `'½'` |
| `isalnum()` | letters or digits |
| `isspace()` | all whitespace |
| `isupper()` / `islower()` | cased chars all upper/lower |
| `isidentifier()` | valid Python variable name |

---

## 7. Formatting strings

### 7.1 f-strings (preferred, Python 3.6+)

In [93]:
name, age = "Asha", 25
f"{name} is {age}"

'Asha is 25'

In [94]:
f"{age + 5}"

'30'

In [95]:
f"{2 + 3 = }"    

'2 + 3 = 5'

---

## 8. Escape sequences and raw strings

| Escape | Meaning |
|---|---|
| `\n` | newline |
| `\t` | tab |
| `\\` | backslash |
| `\'` `\"` | quote |
| `\u20ac` | Unicode code point (4 hex digits) → `€` |
| `\U0001F600` | Unicode code point (8 hex digits) → `😀` |
| `\N{SNOWMAN}` | by Unicode name → `☃` |

**Raw strings** (`r"..."`) turn off escape processing — essential for regex and Windows paths:

In [96]:
print("C:\new\table")    # prints a newline and a tab — wrong!

C:
ew	able


In [97]:
print(r"C:\new\table")   # C:\new\table

C:\new\table


> Gotcha: a raw string cannot end with an odd number of backslashes: `r"abc\"` is a syntax error.

---

## 9. Unicode, code points and encodings

### 9.1 Three layers you must keep separate

```
  What you see        What Python stores        What goes to disk / network
  (glyph)             (code point, a number)    (bytes, via an ENCODING)

     €        ──►        U+20AC (8364)   ──encode('utf-8')──►  b'\xe2\x82\xac'
                                         ◄──decode('utf-8')──
```

In [98]:
ord("€")          # 8364
chr(8364)         # '€'
"€".encode("utf-8")           # b'\xe2\x82\xac'   (3 bytes)
b'\xe2\x82\xac'.decode("utf-8")  # '€'

'€'

- `str` = text (code points).  `bytes` = raw bytes.
- **Encode** when leaving Python (writing files, sending over a network).
- **Decode** when entering Python.
- Always pass an encoding when opening text files: `open(path, encoding="utf-8")`.



Use `casefold()` instead of `lower()` for case-insensitive comparisons.

---

## 10. Internals: how CPython stores a string

### 10.1 The object header

Every `str` object in CPython is a C struct. Simplified:

```
 PyASCIIObject / PyCompactUnicodeObject
 ┌──────────────────────────────┐
 │ ob_refcnt   (reference count)│  ← how many names point here
 │ ob_type     (→ str type)     │
 │ length      (# code points)  │  ← len() is O(1): just read this field
 │ hash        (cached, or -1)  │  ← computed once, reused (§12)
 │ state bits:                  │
 │   kind  (1, 2 or 4 bytes/ch) │  ← PEP 393 flexible storage
 │   ascii (is it pure ASCII?)  │
 │   interned?                  │
 │ ...                          │
 ├──────────────────────────────┤
 │ character data ...  + '\0'   │  ← stored right after the header
 └──────────────────────────────┘
```


### 11.2 Interning

**Interning** = keeping only one copy of a given string value in a global table, and reusing it.

CPython automatically interns:
- Identifiers (variable, function, attribute names).
- String **literals in your code** that look like identifiers (letters, digits, underscore).
- Many compile-time constants.

Strings **created at runtime** (from `join`, input, file reads, concatenation of variables) are generally **not** interned:

## 11. Internals: interning and identity

### 11.1 `==` vs `is`

- `==` compares **values** (same characters).
- `is` compares **identity** (same object in memory).

### 11.2 Interning

**Interning** = keeping only one copy of a given string value in a global table, and reusing it.

CPython automatically interns:
- Identifiers (variable, function, attribute names).
- String **literals in your code** that look like identifiers (letters, digits, underscore).
- Many compile-time constants.

Strings **created at runtime** (from `join`, input, file reads, concatenation of variables) are generally **not** interned:

In [1]:
a = 'Hello'
b = 'Hello'
a == b

True

In [2]:
a is b

True

In [3]:
a = 'hello'
x = "".join(['hel','lo'])
print(x)
a==x

hello


True

In [4]:
a is x

False

In [5]:
a = "hello"
x = "".join(["hel", "lo"])
x == a        # True
x is a        # False  ← different objects, same value

import sys
sys.intern(x) is a    # True  ← manually interned → the shared copy

True

### Visual

```
 Interned table:  { "hello" ──► obj#1 }

 a ─────────────────────────────► obj#1  "hello"
 x ─────────────────────────────► obj#2  "hello"   (built at runtime)
 sys.intern(x) ─────────────────► obj#1
```

### 11.3 Why interning matters

- Dict lookups by attribute/variable names become faster (identity check before full comparison).
- Saves memory when the same string value appears many times (e.g. parsing millions of rows with repeated category names → `sys.intern` them).

> ⚠️ **Never use `is` to compare strings in real code.** Whether two equal strings are the same object is an implementation detail and changes between versions and contexts. Always use `==`.

---

## 12. Internals: hashing

In [6]:
hash("abc")

758726020588553319

- Strings are hashable **because** they are immutable.
- The hash is computed **once** and **cached** in the object header — reusing a string as a dict key many times is cheap.
- CPython uses **SipHash** with a **random seed per process** (hash randomization, a defence against hash-flooding attacks). So `hash("abc")` differs between runs, but is stable *within* one run. Control with the `PYTHONHASHSEED` environment variable.

### How a dict lookup uses this

```
 d["name"]
   1. hash("name")        → cached in the string, O(1) after first time
   2. jump to bucket      → O(1)
   3. same object? (is)   → fast path, done        ← interning helps here
   4. else compare ==     → O(k) for k characters
```

---

## 13. Internals: compile-time constant folding

The compiler pre-computes simple constant expressions. Check with `dis`:

In [7]:
import dis
dis.dis(compile("s = 'a' + 'b'", "", "exec"))

  0           RESUME                   0

  1           LOAD_CONST               2 ('ab')
              STORE_NAME               0 (s)
              LOAD_CONST               1 (None)
              RETURN_VALUE


```
LOAD_CONST   0 ('ab')     ← already joined at compile time!
STORE_NAME   0 (s)
```

So `"a" + "b"` in source code costs nothing at runtime. Very long results are not folded (to avoid bloating `.pyc` files).

---

## 14. Performance: building strings efficiently

### 14.1 The classic problem

Because strings are immutable, `s = s + piece` in a loop *could* copy the whole string every time:

```
 iteration 1:  copy 1 char
 iteration 2:  copy 2 chars
 ...
 iteration n:  copy n chars
 total ≈ n²/2 copies   → O(n²)
```

### 14.2 CPython's hidden optimization

If the string on the left has **only one reference**, CPython resizes it **in place** for `s += x`. That is why this is often fast in CPython:

In [8]:
s = ""
for i in range(10000):
    s += "x"      # usually fast in CPython (in-place resize trick)

But this is **fragile**: it breaks if another name references `s`, and other Python implementations (PyPy, etc.) don't guarantee it.



---

## 15. Concept / interview Q&A

**Q: Why are strings immutable in Python?**
So they can be hashed (dict keys/sets), safely shared between names, interned, and read across threads without locks.

**Q: What does `len()` cost on a string? Why?**
O(1). The length is stored in the object header.

**Q: Why is `s[i]` O(1) when Python supports all of Unicode?**
PEP 393: every string uses one fixed width (1, 2 or 4 bytes per char) chosen from its widest character, so position `i` is a direct offset.

**Q: `a == b` is True but `a is b` is False — how?**
Equal values, different objects. Only some strings (identifiers, identifier-like literals, `sys.intern`-ed strings) are shared.

**Q: When would you use `sys.intern`?**
When the same string values repeat massively (e.g. millions of parsed rows with repeated keys) — saves memory and speeds up dict lookups.

**Q: Why prefer `"".join()` over `+=` in a loop?**
`join` is guaranteed O(n). `+=` is O(n²) in general; CPython's in-place trick is an implementation detail.

**Q: Difference between `str` and `bytes`?**
`str` holds Unicode code points (text). `bytes` holds raw 0–255 values. Convert with `.encode()` / `.decode()`.

**Q: `lower()` vs `casefold()`?**
`casefold()` is more aggressive (e.g. `ß` → `ss`) and is the right choice for case-insensitive matching.

**Q: Is `hash("abc")` the same every run?**
No — hash randomization (SipHash with a per-process seed). Stable within one process.

**Q: What does `"abc"[:]` return?**
The same object (no copy needed because strings are immutable).

---

## 21. Summary

```
 BASICS        literals · indexing · slicing · operators · methods · f-strings
    │
 UNICODE       code points (str)  ⇄  encode/decode  ⇄  bytes · normalization · casefold
    │
 INTERNALS     header (refcount, length, cached hash, kind)
               PEP 393: 1 / 2 / 4 bytes per char → O(1) indexing
               interning · hash randomization · constant folding
    │
 PERFORMANCE   immutable → build with join · know your Big-O
    │
 PROBLEMS      two pointers · hashing/Counter · sliding window · stack
               expand-around-center · DP · KMP / rolling hash
```

**Patterns to recognize in string problems:**

| If the problem says… | Reach for… |
|---|---|
| palindrome, reverse, compare both ends | two pointers |
| anagram, frequency, "count of each" | `Counter` / dict |
| longest / shortest substring with a condition | sliding window |
| brackets, nesting, undo | stack |
| transform one string to another, min operations | dynamic programming |
| find pattern fast in long text | KMP / Rabin–Karp |

In [9]:
# Create a string made of the first, middle, and last character
str1 = "James"
res = str1[0] + str1[len(str1)//2] + str1[-1]
print(res)

Jms


In [12]:
# Create a string made of the middle three characters

str1 = "JhonDipPeta"
res = str1[(len(str1)//2)-1] +str1[(len(str1)//2)]+ str1[(len(str1)//2)+1]
print(res)

Dip


In [13]:
# Reverse a given string
str1[::-1]

'atePpiDnohJ'

In [14]:
#  Split a string on hyphens
str1 = "Emma-is-a-data-scientist"
res = str1.split("-")
print(res)

['Emma', 'is', 'a', 'data', 'scientist']


In [15]:
# Find all occurrences of a substring in a given string by ignoring the case
str1 = "Welcome to USA. usa awesome, isn't it?"
sub_string = "USA"
temp = str1.lower()
count = temp.count(sub_string.lower())
print(count)

2


In [18]:
# balanced test
s1 = 'ynf'
s2 = 'pynative'
for ch in s1:
    if ch not in s2:
        flag = False
        print("Not balanced")
        break
else:
    print("balanced")
        

Not balanced


In [20]:
str1 = "Hello World"
vowels = "aeiouAEIOU"
count = 0

for char in str1:
    if char in vowels:
        count += 1

print("Vowel Count:", count)

Vowel Count: 3


In [22]:
str1 = 'p y t h o n'
res = str1.replace(" ",'')
print(res, type(res))

python <class 'str'>


In [23]:
str1 = 'PyNaTive'
lower = []
upper = []
for char in str1:
    if char.islower():
        lower.append(char)
    else:
        upper.append(char)
res = ''.join(lower + upper)

In [24]:
print(res)

yaivePNT


In [25]:
str1 = "PYnative29@#8496"
total_sum = 0
cnt = 0

for char in str1:
    if char.isdigit():
        total_sum += int(char)
        cnt += 1

# average = sum / count of digits
avg = total_sum / cnt

print("Sum is:", total_sum, "Average is", avg)

Sum is: 38 Average is 6.333333333333333


In [27]:
str1 = "Emma25 is Data scientist50 and AI Expert"
print("The original string is:", str1)

# Split string into individual words
res = []
temp = str1.split()

for item in temp:
    if any(char.isalpha() for char in item) and any(char.isdigit() for char in item):
        res.append(item)
print("Words with alphabets and numbers")
for i in res:
    print(i)

The original string is: Emma25 is Data scientist50 and AI Expert
Words with alphabets and numbers
Emma25
scientist50
